In [1]:
import ollama
# web_tool.py

import json
import socket
import ipaddress
from urllib.parse import urlparse

import requests
from bs4 import BeautifulSoup
from ddgs import DDGS



In [19]:

def is_safe_url(url: str) -> bool:
    """Basic protection against private/internal network URLs."""

    try:
        parsed = urlparse(url)

        if parsed.scheme not in ("http", "https"):
            return False

        hostname = parsed.hostname

        if not hostname:
            return False

        if hostname.lower() in ("localhost",):
            return False

        addresses = socket.getaddrinfo(
            hostname,
            None,
            type=socket.SOCK_STREAM
        )

        for address in addresses:
            ip = ipaddress.ip_address(address[4][0])

            if not ip.is_global:
                return False

        return True

    except Exception:
        return False


def read_webpage(url: str, max_chars: int = 6000) -> str:

    if not is_safe_url(url):
        return "Blocked unsafe URL."

    try:
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 Chrome/120 Safari/537.36"
            )
        }

        response = requests.get(
            url,
            headers=headers,
            timeout=10,
            allow_redirects=True
        )

        # Validate the final URL after redirects as well.
        if not is_safe_url(response.url):
            return "Blocked unsafe redirect."

        response.raise_for_status()

        content_type = response.headers.get(
            "Content-Type", ""
        ).lower()

        if "html" not in content_type:
            return "This page is not HTML content."

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove elements that usually contain irrelevant content.
        for tag in soup([
            "script",
            "style",
            "nav",
            "footer",
            "header",
            "aside",
            "noscript",
            "svg",
            "form"
        ]):
            tag.decompose()

        title = (
            soup.title.get_text(" ", strip=True)
            if soup.title else "Untitled"
        )

        text = soup.get_text(
            separator=" ",
            strip=True
        )

        text = " ".join(text.split())

        return (
            f"Page title: {title}\n"
            f"URL: {response.url}\n"
            f"Content:\n{text[:max_chars]}"
        )

    except Exception as e:
        return f"Failed to read webpage: {str(e)}"


def web_search_and_read(query: str,num_of_search_pages: int =3) -> str:
    """
    Search the web for the user query, read relevant pages,
    and return the extracted content.
    """

    try:
        search_results = DDGS().text(
            query,
            max_results=num_of_search_pages
        )

        search_results = list(search_results)

    except Exception as e:
        return f"Search failed: {str(e)}"

    if not search_results:
        return "No search results found."

    output = []

    # Read the first 3 search results.
    for result in search_results[:num_of_search_pages]:

        title = result.get("title", "Untitled")
        url = result.get("href", "")
        snippet = result.get("body", "")

        if not url:
            continue

        page_content = read_webpage(url)

        output.append({
            "title": title,
            "url": url,
            "search_snippet": snippet,
            "page_content": page_content
        })

    return json.dumps(
        output,
        indent=2,
        ensure_ascii=False
    )

In [10]:
search_query = "Python programming language"
search_results = web_search_and_read(search_query)

In [11]:
print(search_results)

[
  {
    "title": "Python (programming language)",
    "url": "https://en.wikipedia.org/wiki/Python_(programming_language)",
    "search_snippet": "Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, an extensive (\"batteries-included\") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5, capabilities and keywords for typing were added to the language, allowing optional static typing. As of 2026, the Python Software Foundation supports Python 3.15.2, 3.14.10, 3.13.18, 3.12.16 and 3.11.18 that all were \"expedited security 

In [12]:
search_query = "how to build MCP servers"

search_results = web_search_and_read(query=search_query,num_of_search_pages=5)

In [13]:
print(search_results)

[
  {
    "title": "Build an MCP server - Model Context Protocol",
    "url": "https://modelcontextprotocol.io/docs/2026-07-28/develop/build-server",
    "search_snippet": "Jul 28, 2026 · Get started building your own server to use in Claude for Desktop and other clients.",
    "page_content": "Page title: Build an MCP server - Model Context Protocol\nURL: https://modelcontextprotocol.io/docs/2026-07-28/develop/build-server\nContent:\nBuild an MCP server - Model Context Protocol Documentation Index Fetch the complete documentation index at: /llms.txt Use this file to discover all available pages before exploring further. Skip to main content Assistant Responses are generated using AI and may contain mistakes. In this tutorial, we’ll build a simple MCP weather server and connect it to a host, Claude for Desktop. What we’ll be building We’ll build a server that exposes two tools: get_alerts and get_forecast . Then we’ll connect the server to an MCP host (in this case, Claude for Desktop)

In [38]:
search_query = "what is an mcp server, how it works and how to build MCP servers"

search_results = web_search_and_read(query=search_query,num_of_search_pages=10)

In [39]:
print(search_results)

[
  {
    "title": "What Is an MCP Server? How It Works, Examples & Setup",
    "url": "https://mcpservers.org/learn/what-is-an-mcp-server",
    "search_snippet": "Jul 19, 2026 · What is an MCP server? Learn how MCP connects AI clients to tools and data, how local and remote servers work, setup examples, and security checks.",
    "page_content": "Page title: What Is an MCP Server? How It Works, Examples & Setup\nURL: https://mcpservers.org/learn/what-is-an-mcp-server\nContent:\nWhat Is an MCP Server? How It Works, Examples & Setup The short answer An MCP server gives an AI application a standard way to use an outside capability. An MCP server is a program that exposes tools, data, or reusable workflows to an AI application through the Model Context Protocol. Instead of building a different integration for every AI product, a service can implement MCP once and be discovered by compatible clients. The server does not have to be a large cloud service. It can be a local command that reads

In [40]:
tools = [{
    "type":"function",
    "function":{
        "name":"web_search_and_read",
        "description": (
            "Search the internet for information requested by the user. "
            "Search relevant websites, read their pages, and return "
            "the extracted information and URLs. Use this tool when "
            "the answer requires web research, current information, "
            "documentation, news, or information from websites."
        ),
        "parameters":{
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": (
                            "A specific web search query based on "
                            "the user's information requirement."
                        )
                    }
                },
            "required": ["query"]
        }
    }
}]

tool_functions = {
    "web_search_and_read": web_search_and_read
}

In [47]:
def chat(
    user_input: str,
    model: str = "llama2",
    temperature: float = 0.7,
    max_tokens: int = 500,
    tools: list = tools,
    tool_functions: dict = tool_functions
) -> str:
    """
    Chat with the user, using the specified model and tools.
    """

    try:
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": user_input}],
            options={
                "temperature": temperature,
                'num_predict': 10806,    # Max tokens for the model's generated response
                'num_ctx': 6806       # Size of the total memory/prompt context window
            },
            tools=tools
        )
        
        if response.message.tool_calls:
            tool_name = response.message.tool_calls[0].name
            tool_args = response.message.tool_calls[0].arguments

            if tool_name in tool_functions:
                tool_function = tool_functions[tool_name]
                tool_result = tool_function(**tool_args)

                # Send the tool result back to the model for further processing.
                response = ollama.chat(
                    model=model,
                    messages=[
                        {"role": "user", "content": user_input},
                        {"role": "assistant", "content": f"Tool {tool_name} returned: {tool_result}"}
                    ],
                    temperature=temperature,
                    max_tokens=max_tokens,
                    tools=tools,
                    tool_functions=tool_functions
                )

        return response.message.content

    except Exception as e:
        return f"Chat failed: {str(e)}"

In [50]:
chat_input = "What is an MCP server in AI agents, how does it work, and how can I build one?"
chat_response = chat(chat_input,model="deepseek-r1:8b",temperature=0.0,max_tokens=8000,tools=tools,tool_functions=tool_functions)
print(chat_response)


Okay, let's clarify what you mean by "MCP Server." The acronym "MCP" has several meanings depending on the context.

The most likely interpretations related to AI agents are:

1.  **Multiplayer Control Protocol (or similar)**: This could refer to protocols or systems designed for communication and coordination between multiple AI agents in a distributed system, often used in multi-agent environments.
2.  **Minecraft Protocol**: In some contexts, especially within the Minecraft modding scene, "MCP" refers to the official Minecraft modding environment (often contrasted with Forge). However, an "MCP Server" isn't standard terminology here; it usually means a server running the game client or specific mods.
3.  **Mod Configurator Platform**: This is another interpretation from the Minecraft modding community, referring to platforms like MCP-Holograms that allow configuration of in-game elements (like holograms). Again, "server" implies something different.

**To give you accurate informat

In [52]:
response = ollama.chat(
    model="deepseek-r1:8b",
    messages=[{"role": "user", "content": chat_input}],
    options={
        "temperature": 0.0,
        'num_predict': 8000,    # Max tokens for the model's generated response
        'num_ctx': 6806       # Size of the total memory/prompt context window
    },
    tools=tools
)

In [53]:
print(response.message.content)


Okay, let's break down what an "MCP Server" means specifically within the context of **AI Agents**, as opposed to its more common meaning in gaming (Multiplayer Communication Protocol).

In AI agent terminology, an MCP server refers to a system designed to facilitate **communication and coordination between multiple independent AI agents**. It acts as a central hub or intermediary for message passing.

Here's a breakdown:

1.  **What is it?**
    *   An MCP Server is not necessarily the same thing as the game servers mentioned earlier.
    *   In AI, "MCP" often stands for **Multi-Agent Communication Protocol** Server (or sometimes just an Agent Communication Layer server).
    *   It's a platform or service that allows different AI agents (often developed by different entities) to send messages and interact with each other.

2.  **How does it work?**
    *   **Centralized Hub:** The most common interpretation is a centralized server where all inter-agent communication happens.
    * 